# Data Skew: Detection, Diagnostics, and Production Remediation

**Module**: 05_partitions_and_shuffle  
**Notebook**: 06_data_skew_and_solutions.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Detecting data skew via task duration variance, salting skewed join keys, and enabling AQE skew join optimization.
- **Why does this topic exist?**: Data skew is the single most common cause of straggler tasks and OOM crashes in distributed systems.
- **What problem does it solve?**: 1 task takes 1 hour while 99 tasks take 2 seconds.
- **Where is it used in real-world Data Engineering?**: Joining transactions with massive merchant keys or default customer IDs.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Shuffle internals, joins, aggregations, Spark UI |
| **Required depth** | Working knowledge: can detect skew, explain its cost, and choose a remediation |
| **Data Engineer** | Detect skewed keys and apply salting, broadcast, or isolation techniques. |
| **Senior Data Engineer** | Choose a remediation that preserves correctness and fits the data distribution. |
| **Spark specialist** | Reason about AQE skew join handling, salting costs, and hot-key isolation. |
| **Next topic** | `06_memory_and_storage/01_spark_memory_architecture.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Skew means a few partitions carry most of the data. The job's wall-clock time is set by the slowest task, so
one hot key can make a 100-partition stage behave like a single-threaded job. Skew is a data-distribution problem,
not just a Spark setting, and the right fix depends on whether the hot key is legitimate, whether one side is small,
and whether the result can tolerate a modified key.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Skew** | Uneven partition sizes | A few keys dominate shuffle read or task duration |
| **Hot key** | A very frequent key | Common in nulls, default values, and popular entities |
| **Salting** | Add a random suffix to spread a key | Must be reversed or aggregated in a second step |
| **Broadcast join** | Send the small side to every executor | Avoids the shuffle entirely |
| **AQE skew join** | Runtime split of oversized partitions | Requires AQE and eligible join shapes |
| **Isolation** | Process hot keys separately | Often the cleanest fix for a known small set of keys |

### How it works, step by step

1. The job runs a shuffle and creates one partition per reduce key range or hash bucket.
2. A hot key hashes to one partition, which reads far more records than its peers.
3. The stage waits for the slowest task; other executors sit idle.
4. AQE can detect oversized partitions and split them when the join shape allows.
5. Salting changes the key distribution so the hot key spreads across tasks, then a second aggregation combines partial results.
6. Broadcasting removes the shuffle when the build side fits in executor memory.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. The Data Skew Problem

```
Normal Distribution (Balanced Partitions):
Partition 0: [10,000 rows] -> Task 0 (2s) ===> FINISHED!
Partition 1: [10,000 rows] -> Task 1 (2s) ===> FINISHED!
Partition 2: [10,000 rows] -> Task 2 (2s) ===> FINISHED!

Data Skew (Hotspot on'Null'or Popular Key):
Partition 0: [1,000 rows] -> Task 0 (0.2s) ===> IDLE!
Partition 1: [1,000 rows] -> Task 1 (0.2s) ===> IDLE!
Partition 2: [5,000,000 rows] -> Task 2 (45 mins / OOM Crash!) ===> STRAGGLER!
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Data_Skew_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Create artificially skewed dataset (90% of rows have key 'HOTSPOT')
skewed_data = [("HOTSPOT", i) for i in range(100000)] + [(f"KEY_{i}", i) for i in range(10000)]
skewed_df = spark.createDataFrame(skewed_data, ["key", "val"])

print("Top keys in skewed dataset:")
skewed_df.groupBy("key").count().orderBy(F.desc("count")).show(3)

---

## Experiment: Remediating Skew via Key Salting

- **1. What are we testing?**: Salt the skewed key with random integers 0-3 to split the hotspot across 4 partitions.
- **2. What do we expect?**: The hotspot key is divided evenly into 4 distinct sub-keys.

In [ ]:
SALT_FACTOR = 4
salted_df = skewed_df.withColumn("salted_key", F.concat(F.col("key"), F.lit("_"), F.floor(F.rand() * SALT_FACTOR)))
print("Salted key distribution:")
salted_df.groupBy("salted_key").count().orderBy(F.desc("count")).show(5)

**3. What happened?**: The 100,000 rows for'HOTSPOT'were split into 4 balanced partitions of ~25,000 rows each.  
**4. Why did it happen?**: Salting distributes identical keys across multiple distinct hash slots.  
**5. Practical Engineering Lesson**: Salting eliminates single-task hotspots on skewed joins.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Null-key join explosion

- **Industry**: E-commerce
- **Scenario**: Orders are left-joined to a customer dimension, and millions of orders have a null customer ID.
- **Problem**: All null keys hash to the same partition, creating one enormous task while the rest finish quickly.
- **Why this concept is relevant**: Null keys do not need to be joined; filtering them before the join or handling them separately removes the hot partition.
- **Implementation idea**: Compare task-duration percentiles before and after filtering the null keys.
- **Expected result**: A balanced stage and a documented rule for orphan orders.
- **Alternative**: Replace the null with a sentinel and salt it.
- **Trade-offs**: Filtering is cheap and correct when nulls cannot match; salting adds complexity and a second aggregation.

### Use Case 2: Popular product in a clickstream aggregation

- **Industry**: Digital media
- **Scenario**: A session aggregation is dominated by one viral product ID.
- **Problem**: The hot product lands in one partition and the stage takes hours.
- **Why this concept is relevant**: Salting the product key spreads the hot product across tasks, then a second aggregation sums the partial counts.
- **Implementation idea**: Measure the max task duration and shuffle read before and after salting.
- **Expected result**: A balanced stage with a small second aggregation.
- **Alternative**: Isolate the hot product and compute it separately, then union the results.
- **Trade-offs**: Salting is general but adds a shuffle; isolation is simpler for a known, small hot-key set.

### Use Case 3: Skewed fraud alerts

- **Industry**: Financial services
- **Scenario**: A fraud-detection join is slow because one merchant account generates most transactions.
- **Problem**: The merchant key is genuinely hot; dropping it would lose fraud signal.
- **Why this concept is relevant**: Broadcasting the merchant dimension or isolating the hot merchant preserves the signal while fixing the shuffle.
- **Implementation idea**: Test broadcast if the dimension is small; otherwise isolate the hot merchant and union the results.
- **Expected result**: A correct result with bounded task duration.
- **Alternative**: A dedicated stream for high-volume merchants.
- **Trade-offs**: Isolation keeps the main job predictable; a separate pipeline adds operational overhead.

---

## Business Value

- Predictable SLAs instead of jobs that occasionally take all night.
- Lower compute waste from idle executors.
- Correct results for null and default keys.
- A repeatable playbook for the most common performance escalation.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Look at max/median task duration and shuffle read per task, not just stage averages.
- Check for nulls and sentinel values before inventing a salting scheme.
- Try broadcast first when one side is small.
- Enable AQE and verify whether skew join handling applies to the plan.
- After salting, always verify row counts and aggregation correctness.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Detect and fix a hot key

**Steps**

1. Create a dataset where one key has 80% of the rows.
2. Run a `groupBy` and record the max versus median task duration and shuffle read.
3. Apply salting or isolate the hot key, then repeat the measurement.

**What to observe**: The max task duration dominates before the fix; after the fix, task durations converge.

**Healthy result**: The ratio of max to median duration falls and the stage completes faster.

**Problematic result**: The job still has one slow task because the salt is applied only to one side or the second aggregation is skewed.

**Engineering takeaway**: Skew remediation must be verified at the task level, not just by a successful job.

---

## Trade-offs and Alternatives

### Advantages

- Large latency improvements
- Better cluster utilization
- AQE can handle some cases automatically

### Limitations

- Salting adds a second shuffle
- AQE only handles eligible joins
- Hot keys may be business-critical

### When to use it

- Null-heavy joins
- Viral entities
- Uneven group-bys

### When not to use it

- Salting every key
- Dropping hot keys without a business rule
- Assuming more partitions fix skew

### Alternatives inside Spark

- AQE skew join
- Salting
- Broadcast join
- Hot-key isolation

### Alternatives outside Spark

- Pre-aggregation in a warehouse
- Streaming with key-level parallelism
- Custom partitioners

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Skew in Profiling
**Problem Statement**: Profile orders channel column for skew.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Broadcast Join to Avoid Skew
**Problem Statement**: Explain why broadcast join eliminates skew.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): AQE Skew Join Config
**Problem Statement**: Print AQE skew join configuration parameters.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Salting changed the answer

**Scenario**: After salting a join key, the output row count doubles.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the salt was applied to both sides of the join.
2. Verify the join condition includes the salt column.
3. Confirm the post-aggregation step restores the original grain.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Skew in Profiling):
```python
orders.groupBy('channel').count().orderBy(F.desc('count')).show()
```

#### Solution 2 (Broadcast Join to Avoid Skew):
```python
print('Broadcast join does not shuffle data by key, eliminating key-based skew.')
```

#### Solution 3 (AQE Skew Join Config):
```python
print('skewJoin.enabled:', spark.conf.get('spark.sql.adaptive.skewJoin.enabled','true'))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Data Skew**: Uneven key distribution causing straggler tasks.
- **Remediation**: 1. Broadcast Join 2. Salting 3. AQE Skew Join 4. Isolate Skewed Keys.

### Essential Technical Terms
- `Straggler`: Task running significantly slower than peers.
- `Salting`: Appending random integer to split keys.

### Common Engineering Mistakes
- **Adding more executors to fix skew**: Extra executors do not help if 1 task holds all skewed records.

### Interview Questions on This Topic
- **Q: How do you detect data skew in the Spark UI?**
  - *A*: Compare 75th percentile vs Max task duration in the Stages tab; if Max is orders of magnitude larger, skew is present.


### More Interview Questions

- **Q: How do you detect skew in the Spark UI?**
  - *A*: Compare max and median task duration and shuffle read per task within a stage; a few tasks much larger than the median indicate skew.
- **Q: Why can adding more shuffle partitions fail to fix skew?**
  - *A*: A single hot key still hashes to one partition; increasing the partition count does not split that key's records.

### What to Study Next
Module 05 complete! Proceed to Module 06: Memory Architecture and Storage.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')